Check files uploaded to volume

In [0]:
display(dbutils.fs.ls("/Volumes/traveldbs/bronzedbs/transportationdbs_volume/"))

path,name,size,modificationTime
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/DATABASE_SCHEMA.txt,DATABASE_SCHEMA.txt,3411,1790216550000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/customers.csv,customers.csv,15110,1790216550000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/delivery_events.csv,delivery_events.csv,22240101,1790216564000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/driver_monthly_metrics.csv,driver_monthly_metrics.csv,265289,1790216550000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/drivers.csv,drivers.csv,13009,1790216550000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/facilities.csv,facilities.csv,4323,1790216550000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/fuel_purchases.csv,fuel_purchases.csv,20377087,1790216564000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/loads.csv,loads.csv,8384050,1790216559000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/maintenance_records.csv,maintenance_records.csv,314754,1790216550000
dbfs:/Volumes/traveldbs/bronzedbs/transportationdbs_volume/routes.csv,routes.csv,3075,1790216550000


Read the CSV

In [0]:
df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/traveldbs/bronzedbs/transportationdbs_volume/*.csv")

display(df)

event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,location_state
EVT00000001,LOAD00000001,TRIP00000001,Pickup,FAC00034,2022-01-01 18:00:00.000000,2022-01-01 20:58:55.918185,0,False,Houston,TX
EVT00000002,LOAD00000001,TRIP00000001,Delivery,FAC00046,2022-01-02 23:10:55.918185,2022-01-02 21:30:22.142060,230,True,Detroit,MI
EVT00000003,LOAD00000002,TRIP00000002,Pickup,FAC00015,2022-01-01 18:00:00.000000,2022-01-01 17:37:26.608430,62,True,Kansas City,MO
EVT00000004,LOAD00000002,TRIP00000002,Delivery,FAC00050,2022-01-02 02:13:26.608430,2022-01-02 05:55:46.238257,67,False,Indianapolis,IN
EVT00000005,LOAD00000003,TRIP00000003,Pickup,FAC00001,2022-01-01 08:00:00.000000,2022-01-01 07:28:02.169634,83,True,Columbus,OH
EVT00000006,LOAD00000003,TRIP00000003,Delivery,FAC00022,2022-01-03 04:28:02.169634,2022-01-03 03:05:02.833256,11,True,Portland,OR
EVT00000007,LOAD00000004,TRIP00000004,Pickup,FAC00029,2022-01-01 17:00:00.000000,2022-01-01 19:32:56.020030,50,False,Phoenix,AZ
EVT00000008,LOAD00000004,TRIP00000004,Delivery,FAC00037,2022-01-02 06:38:56.020030,2022-01-02 08:41:06.714927,142,False,Denver,CO
EVT00000009,LOAD00000005,TRIP00000005,Pickup,FAC00046,2022-01-01 08:00:00.000000,2022-01-01 11:51:35.544202,73,False,Houston,TX
EVT00000010,LOAD00000005,TRIP00000005,Delivery,FAC00050,2022-01-02 22:51:35.544202,2022-01-02 22:23:50.185284,236,True,Seattle,WA


Read csv files + APPEND ,Write Bronze Delta table

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

volume_path = "/Volumes/traveldbs/bronzedbs/transportationdbs_volume"

bronze_schema = "traveldbs.bronzedbs"

files = dbutils.fs.ls(volume_path)

for file in files:

    if file.name.endswith(".csv"):

        table_name = file.name.replace(".csv", "").lower().replace("-", "_")

        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("encoding", "UTF-8")
            .csv(file.path)
        )

        # APPEND into Bronze Delta table
        df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(f"{bronze_schema}.{table_name}")

        print(f"Bronze created/updated: {table_name}")

Bronze created/updated: customers
Bronze created/updated: delivery_events
Bronze created/updated: driver_monthly_metrics
Bronze created/updated: drivers
Bronze created/updated: facilities
Bronze created/updated: fuel_purchases
Bronze created/updated: loads
Bronze created/updated: maintenance_records
Bronze created/updated: routes
Bronze created/updated: safety_incidents
Bronze created/updated: trailers
Bronze created/updated: trips
Bronze created/updated: truck_utilization_metrics
Bronze created/updated: trucks


In [0]:
spark.sql("SHOW TABLES IN traveldbs.bronzedbs").show(truncate=False)

+---------+-------------------------+-----------+
|database |tableName                |isTemporary|
+---------+-------------------------+-----------+
|bronzedbs|customers                |false      |
|bronzedbs|delivery_events          |false      |
|bronzedbs|driver_monthly_metrics   |false      |
|bronzedbs|drivers                  |false      |
|bronzedbs|facilities               |false      |
|bronzedbs|fuel_purchases           |false      |
|bronzedbs|loads                    |false      |
|bronzedbs|maintenance_records      |false      |
|bronzedbs|routes                   |false      |
|bronzedbs|safety_incidents         |false      |
|bronzedbs|trailers                 |false      |
|bronzedbs|trips                    |false      |
|bronzedbs|truck_utilization_metrics|false      |
|bronzedbs|trucks                   |false      |
+---------+-------------------------+-----------+

